# Hyperparameters Tuning

<!-- TOC -->
* [Hyperparameters Tuning](#hyperparameters-tuning)
* [🏁 1. Initialization](#-1-initialization)
    * [1.0 Installing dependencies](#10-installing-dependencies)
    * [1.1 Importing Libraries](#11-importing-libraries)
    * [1.2 Global Definitions](#12-global-definitions)
    * [1.3 Global Settings](#13-global-settings)
    * [1.4 Global Structure](#14-global-structure)
    * [1.5 Global Function Definitions](#15-global-function-definitions)
* [📂 2. Dataset](#-2-dataset)
    * [2.0 Acquire Dataset](#20-acquire-dataset)
    * [2.1 Split dataset](#21-split-dataset)
    * [2.2 Convert to file images](#22-convert-to-file-images)
* [📚️ 3. Models Settings](#-3-models-settings)
    * [3.1 Model selection](#31-model-selection)
    * [3.2 Training Configuration](#32-training-configuration)
* [🔍️ 4. Hyperparameter Tuning](#-4-hyperparameter-tuning)
  * [4.1 Genetic Algorithm](#41-genetic-algorithm)
  * [4.2 K-Fold Cross Validation](#42-k-fold-cross-validation)
      * [4.2.1 Load labels and classes](#421-load-labels-and-classes)
      * [4.2.2 Init empty pandas DataFram](#422-init-empty-pandas-datafram)
      * [4.2.3 Count the instances of each class-label present in the annotation files.](#423-count-the-instances-of-each-class-label-present-in-the-annotation-files)
      * [4.2.4 Configure K-Fold Dataset Split](#424-configure-k-fold-dataset-split)
      * [4.2.5 Do something](#425-do-something)
      * [4.2.6 Do something](#426-do-something)
      * [4.2.7 Create the directories](#427-create-the-directories)
      * [4.2.8 Copy the files](#428-copy-the-files)
<!-- TOC -->

# 🏁 1. Initialization

### 1.0 Installing dependencies

In [ ]:
!pip install ultralytics markdown rich wrapt pandas huggingface_hub scikit-learn opencv-python wandb python-dotenv datasets -q

### 1.1 Importing Libraries

In [ ]:
from datasets import load_dataset, Image, concatenate_datasets, DatasetDict
from sklearn.model_selection import KFold
from ultralytics import YOLO, settings
from typing import Iterable, Union
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv
from datetime import datetime
from itertools import chain
from pathlib import Path
from PIL import Image


from tqdm import tqdm
import pandas as pd
import ultralytics
import numpy as np
import datetime
import random
import shutil
import wandb
import uuid
import yaml
import math
import re
import os

ultralytics.checks()

### 1.2 Global Definitions

- **DATASET_ROOT_DIR**: Path of the dataset, directory could be empty or contain images and labels.
- **DATASET_SPLIT_NAME**: Name of the split for Hugging Face. For dataset creation it should have only one split.
- **MODELS_DIRECTORY**: Directory where models are stored.

In [ ]:
DATASET_ROOT_DIR = Path('./datasets/main')
DATASET_SPLIT_NAME = DATASET_ROOT_DIR / 'train_validation_test'
IMAGE_APPLY_GRAYSCALE = True

# You may not need to modify these variables
TRAINING_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'train'
VALIDATION_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'valid/'
TEST_DATASET_DIRECTORY = DATASET_ROOT_DIR / 'test/'

IMAGES_EXTENSIONS = [".jpg", ".jpeg", ".JPG", ".JPEG"]
MODELS_DIRECTORY = Path('./models/')
load_dotenv()

### 1.3 Global Settings

Log in to the different providers.

- **Hugging Face**: To download and upload dataset
- **Weights & Biases**: To keep track of the different training results

In [ ]:
# YOLO settings
settings.update({"wandb": False})

# Initialize Weights & Biases environment
# wandb.login(key=os.getenv("WANDB_TOKEN"))

# login("TOKEN") # Keep commented if token loaded from .env file

### 1.4 Global Structure

`dataset_structure` is used to simplify files process through the differents steps. It will be filled automatically later. You don't need to change anything.

In [ ]:
dataset_structure = {
    "root": Path(""),
    "name": "",
    "classes": [],
    "train": {
        "images": [],
        "labels": [],
    },
    "valid": {
        "images": [],
        "labels": [],
    },
    "test": {
        "images": [],
        "labels": [],
    },
}


### 1.5 Global Function Definitions

In [ ]:
def list_files(
        directory: Union[str, Path],
        extensions: Iterable[str],
        include_root_directory: bool = False,
        recursive: bool = False,
) -> list[Path]:
    directory = Path(directory)
    extensions = tuple(extensions)

    matched_files = []

    if recursive:
        iterator = directory.rglob("*")
    else:
        iterator = directory.iterdir()

    for p in iterator:
        if p.is_file() and p.suffix in extensions:
            matched_files.append(
                p if include_root_directory else p.name
            )

    return matched_files


def numeric_key(name):
    """Extract the first number from a filename for sorting."""
    nums = re.findall(r'\d+', name)
    return int(nums[0]) if nums else float('inf')


def sort_files_by_number(files_to_sort: list):
    """
    Sort a list of filenames by the first number found in each name.

    Args:
        files_to_sort (list): List of filenames (strings)

    Returns:
        list: Sorted list of filenames
    """
    return sorted(files_to_sort, key=lambda i: int(i.stem))


def update_dataset_structure():
    dataset_structure["root"] = Path(DATASET_ROOT_DIR)
    dataset_structure["name"] = DATASET_ROOT_DIR.name

    dataset_structure["train"]["images"] = sort_files_by_number(
        list_files(TRAINING_DATASET_DIRECTORY, IMAGES_EXTENSIONS, True))
    dataset_structure["train"]["labels"] = sort_files_by_number(list_files(TRAINING_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["valid"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, IMAGES_EXTENSIONS, True))
    dataset_structure["valid"]["labels"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["test"]["images"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, IMAGES_EXTENSIONS, True))
    dataset_structure["test"]["labels"] = sort_files_by_number(list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True))

    dataset_structure["classes"] = ["drone", "other"]


def delete_files_in_dataset(files_to_delete: list):
    try:
        confirm = input("Files are going to be deleted. Type 'yes' to continue: ").strip().lower()
        if confirm != 'yes':
            print("Deletion aborted by user.")
            return

        for file in files_to_delete:
            if os.path.isfile(file):
                os.remove(file)
                print(f"Deleted: {file}")
            else:
                print(f"Warning: File does not exist: {file}")

    except KeyboardInterrupt:
        print("\nDeletion aborted by user (KeyboardInterrupt).")
    finally:
        try:
            update_dataset_structure()
        except NameError:
            pass


def backup_dataset():
    dataset_path = dataset_structure.get("path", "")
    backup_dir = os.path.join(dataset_path, "backup")
    os.makedirs(backup_dir, exist_ok=True)

    now = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    target_name = f"{dataset_structure["name"]}-{now}"
    target_path = os.path.join(backup_dir, target_name)

    def ignore_backup(_, names):
        return {"backup"} if "backup" in names else set()

    shutil.copytree(dataset_path, target_path, ignore=ignore_backup)
    print(f"Backup created at: {target_path}")
    return str(target_path)


def interpret_map(map_value: float) -> str:
    """
    Interpret mAP value according to standard object detection heuristics.
    """
    if map_value < 0.10:
        return "Model is effectively failing"
    elif map_value < 0.30:
        return "Very weak performance"
    elif map_value < 0.50:
        return "Usable baseline"
    elif map_value < 0.70:
        return "Good performance"
    else:
        return "Strong performance"


def plot_image_grid(images_path, nb_cols=4, max_images_preview=-1, show_title=False):
    if max_images_preview != -1:
        images_path = images_path[:max_images_preview]

    rows = math.ceil(len(images_path) / nb_cols)

    img = Image.open(images_path[0])
    w, h = img.size  # pixels

    dpi = 100

    img_w = w / dpi
    img_h = h / dpi
    #
    plt.figure(figsize=(nb_cols * img_w, rows * img_h))

    for i, path in enumerate(images_path):
        img = Image.open(path)
        plt.subplot(rows, nb_cols, i + 1)
        plt.imshow(img)
        plt.axis("off")
        if show_title:
            plt.title(path.name, fontsize=25)

    plt.tight_layout()
    plt.show()

def create_run_name(version, size):
    Path("./runs").mkdir(parents=False, exist_ok=True)
    session_id = str(uuid.uuid4()).split("-")[0]

    run_name = f"{version}-{size}-{session_id}"

    with open("./runs/runs_history.txt", "a") as f:
        f.write(run_name + "\n")
    return run_name


def last_run_name(offset=0):
    with open("./runs/runs_history.txt", "r") as f:
        return f.read().splitlines()[-offset - 1]

# 📂 2. Dataset

### 2.0 Acquire Dataset

Download the dataset from Hugging Face. [Hibou-Foundation](https://huggingface.co/Hibou-Foundation) is the official repo of the project.

If the dataset is already downloaded and converted to files, go directly to step: **3. Models Settings**

In [ ]:
dataset = load_dataset("Hibou-Foundation/computer-vision")

### 2.1 Split dataset

Split the dataset into train, validation, test.

In [ ]:
base_split = "train_validation_test"
label_column = "class_id"

train_ratio = [0.8, 0.8]  # [class 0, class 1]
valid_ratio = [0.1, 0.1]
test_ratio = [0.1, 0.1]

seed = 42

for i in range(len(train_ratio)):
    assert train_ratio[i] + valid_ratio[i] + test_ratio[i] == 1.0

train_parts = []
valid_parts = []
test_parts = []

num_classes = len(train_ratio)

for cls in range(num_classes):
    cls_ds = dataset[base_split].filter(
        lambda x: x[label_column] == cls
    )

    cls_ds = cls_ds.shuffle(seed=seed)

    n = len(cls_ds)
    n_train = int(n * train_ratio[cls])
    n_valid = int(n * valid_ratio[cls])

    train_parts.append(cls_ds.select(range(0, n_train)))
    valid_parts.append(cls_ds.select(range(n_train, n_train + n_valid)))
    test_parts.append(cls_ds.select(range(n_train + n_valid, n)))

train_ds = concatenate_datasets(train_parts).shuffle(seed=seed)
valid_ds = concatenate_datasets(valid_parts).shuffle(seed=seed)
test_ds = concatenate_datasets(test_parts).shuffle(seed=seed)

dataset = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds,
})
dataset

### 2.2 Convert to file images

Now images and labels must be converted into regular files to be processed by YOLO

In [ ]:
# Create folders
for split in ["train", "valid", "test"]:
    os.makedirs(f"{DATASET_ROOT_DIR}/{split}", exist_ok=True)


def export_to_yolo(ds, split_name):
    for idx, sample in enumerate(ds):
        image = sample["image"]  # already a PIL.Image
        label = sample["raw_label"]  # already YOLO format [[class, cx, cy, w, h], ...]
        img_name = sample["name"]
        txt_name = img_name.split(".")[0] + ".txt"

        # Save image
        img_path = f"{DATASET_ROOT_DIR}/{split_name}/{img_name}"
        image.save(img_path, quality=95)

        # Save labels
        lbl_path = f"{DATASET_ROOT_DIR}/{split_name}/{txt_name}"
        with open(lbl_path, "w") as f:
            f.write(label)


# Run export
split_mapping = {"train": "train", "validation": "valid", "test": "test"}
for hf_split, folder_name in split_mapping.items():
    export_to_yolo(dataset[hf_split], folder_name)
update_dataset_structure()

# 📚️ 3. Models Settings

### 3.1 Model selection

Select the size and the version of the YOLO model to train.

Available Sizes


| Size    | Ideal Use Case                                      |
|---------|-----------------------------------------------------|
| nano    | Resource-constrained devices, high FPS requirements |
| small   | Laptops, edge devices with moderate compute         |
| medium  | Desktop systems, accuracy-focused applications      |
| large   | High-end systems, accuracy-critical applications    |
| x-large | Server/workstation deployment, offline processing   |

**Source**: https://deepwiki.com/niconielsen32/YOLO-3D/4.2-model-size-selection

In [ ]:
selected_size = "nano"
selected_version = "26"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}
model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}.pt"
model_path = MODELS_DIRECTORY / model_name
model = YOLO(model_path, task="detect")

create_run_name(selected_version, selected_size) # Save the run name into a file, so it can be retrieve even after jupyter kernel ended.

### 3.2 Training Configuration
Define hyperparameters (epochs, batch size, image size)

In [ ]:
train_config = {
    'imgsz': 640,
    'batch': 16,
    'epochs': 30,
    'optimizer': 'AdamW',
    'project': 'computer-vision',
    'plots': False,
    'save': False,
    'val': True,
    'device': [0, 1]
}
train_config

# 🔍️ 4. Hyperparameter Tuning

- [Official documentation](https://docs.ultralytics.com/guides/hyperparameter-tuning/)

## 4.1 Genetic Algorithm

In [ ]:
# Define search space
search_space = {
    "lr0": (1e-4, 5e-3),
    "degrees": (0.0, 10.0),
    "scale": (0.4, 1.0),
    "translate": (0.05, 0.2),
    "warmup_epochs": (1.0, 5.0),
    "momentum": (0.85, 0.95),
    "perspective": (0.0, 0.001),
    "mosaic": (0.3, 0.8),
    "mixup": (0.0, 0.2),
    "box": (4.0, 10.0),
    "cls": (0.5, 1.5),
}
resume_tuning = False

In [ ]:
data_yaml = dict(
    train=os.path.join('../../', TRAINING_DATASET_DIRECTORY),
    val=os.path.join('../../', VALIDATION_DATASET_DIRECTORY),
    nc=2,
    channels=1 if IMAGE_APPLY_GRAYSCALE else 3,
    names=['drone', 'other']
)

data_config_path = DATASET_ROOT_DIR / 'data.yaml'

with open(data_config_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)

%cat "$data_config_path"

In [ ]:
model.tune(
    **train_config,
    data=data_config_path,
    iterations=500,
    space=search_space
)

## 4.2 K-Fold Cross Validation

#### 4.2.1 Load labels and classes

In [ ]:
update_dataset_structure()

train_path = TRAINING_DATASET_DIRECTORY
valid_path = VALIDATION_DATASET_DIRECTORY
test_path = TEST_DATASET_DIRECTORY

labels = list(chain(dataset_structure["train"]["labels"], dataset_structure["valid"]["labels"]))

cls_idx = list(range(len(dataset_structure["classes"])))
classes = list(dataset_structure["classes"])

#### 4.2.2 Init empty pandas DataFram

In [ ]:
index = [label.stem for label in labels]  # uses base filename as ID (no extension)
labels_df = pd.DataFrame([], columns=cls_idx, index=index)

#### 4.2.3 Count the instances of each class-label present in the annotation files.

In [ ]:
for label in labels:
    lbl_counter = Counter()

    with open(label) as lf:
        lines = lf.readlines()

    for line in lines:
        # classes for YOLO label use integer at the first position of each line
        lbl_counter[int(line.split(" ", 1)[0])] += 1

    labels_df.loc[label.stem] = lbl_counter

labels_df = labels_df.fillna(0.0).infer_objects(copy=False)  # replace `nan` values with `0.0`
labels_df

#### 4.2.4 Configure K-Fold Dataset Split

In [ ]:
random.seed(0)  # for reproducibility
k_split = 5
kf = KFold(n_splits=k_split, shuffle=True, random_state=20)  # setting random_state for repeatable results

k_folds = list(kf.split(labels_df))

#### 4.2.5 Do something

In [ ]:
folds = [f"split_{n}" for n in range(1, k_split + 1)]
folds_df = pd.DataFrame(index=index, columns=folds)

for i, (train, val) in enumerate(k_folds, start=1):
    col = f"split_{i}"

    folds_df.loc[labels_df.iloc[train].index, col] = "train"
    folds_df.loc[labels_df.iloc[val].index, col] = "val"


#### 4.2.6 Do something

In [ ]:
fold_lbl_distrb = pd.DataFrame(index=folds, columns=cls_idx)

for n, (train_indices, val_indices) in enumerate(k_folds, start=1):
    train_totals = labels_df.iloc[train_indices].sum()
    val_totals = labels_df.iloc[val_indices].sum()

    # To avoid division by zero, we add a small value (1E-7) to the denominator
    ratio = val_totals / (train_totals + 1e-7)
    fold_lbl_distrb.loc[f"split_{n}"] = ratio

#### 4.2.7 Create the directories

In [ ]:
# Initialize an empty list to store image file paths
images = list(chain(dataset_structure["train"]["images"], dataset_structure["valid"]["images"]))

# Create the necessary directories and dataset YAML files
save_path = Path(DATASET_ROOT_DIR / f"{datetime.date.today().isoformat()}_{k_split}-Fold_Cross-val")
save_path.mkdir(parents=True, exist_ok=True)
ds_yamls = []

for split in folds_df.columns:
    # Create directories
    split_dir = save_path / split
    split_dir.mkdir(parents=True, exist_ok=True)
    (split_dir / "train").mkdir(parents=True, exist_ok=True)
    (split_dir / "val").mkdir(parents=True, exist_ok=True)

    # Create dataset YAML files
    dataset_yaml = split_dir / f"{split}_dataset.yaml"
    ds_yamls.append(dataset_yaml)

    with open(dataset_yaml, "w") as ds_y:
        yaml.safe_dump(
            {
                "path": split_dir.as_posix(),
                "train": "train",
                "val": "val",
                "names": classes,
            },
            ds_y,
        )

#### 4.2.8 Copy the files

In [ ]:
for image, label in tqdm(zip(images, labels), total=len(images), desc="Copying files"):
    for split, k_split in folds_df.loc[image.stem].items():
        # Destination directory
        img_to_path = save_path / str(split) / k_split
        lbl_to_path = save_path / str(split) / k_split

        # Copy image and label files to a new directory (SamefileError if a file already exists)
        shutil.copy(image, img_to_path / image.name)
        shutil.copy(label, lbl_to_path / label.name)

In [ ]:
results = {}

for k, dataset_yaml in enumerate(ds_yamls):
    fold_run_name = f"{selected_version}-{selected_size}-{run_session_id}-fold_{k + 1}"

    model = YOLO(model_path, task="detect")
    results[k] = model.train(data=dataset_yaml, name=fold_run_name, **train_config)

In [ ]:
cv_metrics = {}

for k, dataset_yaml in enumerate(ds_yamls):
    model_path = Path(results[k].save_dir) / "weights/best.pt"
    fold_run_name = f"{selected_version}-{selected_size}-{run_session_id}-val-fold_{k + 1}"

    print(fold_run_name)

    validation_model = YOLO(model_path)
    metrics = validation_model.val(data=dataset_yaml, name=fold_run_name, **train_config, plots=True)

    cv_metrics[k] = {
        "mAP50": metrics.box.map50,
        "mAP75": metrics.box.map75,
        "mAP50-95": metrics.box.map,
        "per_class": metrics.box.maps,
    }

In [ ]:
map50 = np.mean([m["mAP50"] for m in cv_metrics.values()])
map75 = np.mean([m["mAP75"] for m in cv_metrics.values()])
map5095 = np.mean([m["mAP50-95"] for m in cv_metrics.values()])

map50_std = np.std([m["mAP50"] for m in cv_metrics.values()])
map75_std = np.std([m["mAP75"] for m in cv_metrics.values()])
map5095_std = np.std([m["mAP50-95"] for m in cv_metrics.values()])

print(f"mAP50:\t\t{map50:.4f} ± {map50_std:.4f} → " f"{interpret_map(float(map50))}")
print(f"mAP75:\t\t{map75:.4f} ± {map75_std:.4f} → "f"{interpret_map(float(map75))}")
print(f"mAP50-95:\t{map5095:.4f} ± {map5095_std:.4f} → " f"{interpret_map(float(map5095))}")